# Models
[Previous Notebook](preprocessing.ipynb) | [**Next Notebook** (TBD)]()

To begin this section, i would like to do some inital analysis of certain models discussed in the course and why they may or may not be a good fit for the problem

## Instance and tree based
### K Nearest Neighbours
As said before, the dataset is very imbalanced (~97% bias towards not depressed). Because of this, I think that voting by neighbourhood majority will not be a very effective means of predicting the target, especially if the individual features have a lot of varience, which i saw evidence of through the `LogisticRegression` tests of the previous notebook (~0.23 `std`).

### Decision Tree
Unlike `KNN`, A `Decision Tree` has the potential to be a very good model for this problem. With a suitible `max_depth` and split criteria, a decision tree can uncover more complex patterns in a dataset and is much more fitted for imbalanced data than KNN. A tree based model also has the capability of ignoring irrelevant or weak features, unlike an instance based model.

## Ensembles
### Random Forest
Although `Random Forest` is built on the back of decision trees, I do not see it being an effective model for this problem. By definition, `Random Forest` chooses a random subset of features for each split while constructing the tree for each bootstrap sample and then aggregate by voting. Fro my understanding of the problem and due to the poor balance of the dataset, i dont think aggregation (voting) will be a fair means of resolving a prediction. 

### Gradient Boost
Building on the predicted effectiveness of a `Decision Tree`, I think `Gradient Boost` will be a better ensemble method for increasing model performance, as it incrementally cancels the errors of the model using the samples loss gradient. From my own intuition, keeping in mind the imbalanced dataset and the logic of the problem, incrementally adjusting the error of decisions in the tree makes more sense than randomly choosing features for each split and aggregating the results at the end.

## Deep Learning
### Multi Layer Perceptron (Neural Network)
It will be interesting to see how a `Neural Network` interacts with this problem. I think it will be challenging to tune the hyperparameters such as layers and learning rate to fit the imbalanced data without under or overfitting.

# To begin...
I will allow importing my modular scripts from [src/](/src/) and split my dataset into training and test sets to avoid leakage.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import pandas as pd
import numpy as np
import environment

I will then split the data into training and testing sets before modelling to avoid leakage. This will also allow me to use the same splits for all models in this section

In [2]:
from get_data_splits import get_data_splits

splits = get_data_splits()
X_train, X_test, y_train, y_test = splits

print(f"Training set: {y_train.shape[0]}, Test set: {y_test.shape[0]}")
print(f"Random State: {environment.Random_State}")

Training set: 840, Test set: 360
Random State: 42


# **\*Come back to this at the end of this notebook\***

[Back](#please-follow-these-steps)

In [3]:
remainder="passthrough"
#remainder="drop"

# KNN

The proceeding model evaluations will follow the same structure. 

1. I will cross validate them and tune hyper parameters using [src/grid_cross_validation](/src/grid_cross_validation.py)
2. I will use the best model from the CV to evaluate the test set using [src/run_test.py](/src/run_test.py)
3. I will write about my interpretation of the results

In [4]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from preprocessor import preprocessor
from grid_cross_validation import grid_cross_validation

knn = make_pipeline(preprocessor(remainder=remainder), StandardScaler(), KNeighborsClassifier()) #scaling required for KNN

knn_cv, best_estimator, best_params = grid_cross_validation(knn, X_train, y_train, param_grid={
    'kneighborsclassifier__n_neighbors': [3, 5, 7, 11, 21],
    'kneighborsclassifier__weights': ['uniform', 'distance'],
    'kneighborsclassifier__p': [1, 2, 3]
})

display(best_params)
print("Best f1 score: ", knn_cv.best_score_)

,Best Parameter
n_neighbors,3
p,2
weights,uniform


Best f1 score:  0.08


### Hyperparameter tuning
| Hyperparameter | Reason of choice |
| -------------- | ---------------- |
| `n_neighbors` | most important hyperparameter for KNN, it decides how many neighbors to cross reference (must be odd) |
| `weights` | does weighting the votes by the distance improve anything? |
| `p` | does the choice of manhattan/euclidean (or higher orders) make a difference? |

In [5]:
from run_test import run_test

knn_results = run_test(best_estimator, splits)
display(knn_results['confusion_matrix'])
print(knn_results['classification_report'])

,predicted not depressed,predicted depressed
actual not depressed,351,0
actual depressed,9,0


Classification Report
               precision    recall  f1-score   support

not depressed       0.97      1.00      0.99       351
    depressed       0.00      0.00      0.00         9

     accuracy                           0.97       360
    macro avg       0.49      0.50      0.49       360
 weighted avg       0.95      0.97      0.96       360



### Interpretation of results

As you can see, `KNN` performs absolutely horribly for my problem, with `0`'s for all positive metrics (testing) and `f1_score=0.08` as the best result from cross validation. This makes sense because, as mentioned, the dataset is so imbalanced and the datapoints likely do not lie very close in feature space. Additionally, the added dimensions due to onehot encoding may skew the distance values, though not substantially.

# Decision Tree

In [6]:
from sklearn.tree import DecisionTreeClassifier

decisionTree = make_pipeline(preprocessor(remainder=remainder), DecisionTreeClassifier(criterion='entropy', random_state=environment.Random_State))

decisionTree_cv, best_estimator, best_params = grid_cross_validation(decisionTree, X_train, y_train, param_grid={
    'decisiontreeclassifier__max_depth': [1, 3, 5, 7, 10],
    'decisiontreeclassifier__max_features': ['sqrt', 'log2', None],
})

display(best_params)
print("Best f1 score: ", decisionTree_cv.best_score_)

,Best Parameter
max_depth,5
max_features,None


Best f1 score:  0.9777777777777779


### Hyperparameter tuning
| Hyperparameter | Reason of choice |
| -------------- | ---------------- |
| `max_depth` | which depth of tree is most suitable? i.e. is the pattern dominated by a lower or higher number of decisions |
| `max_features` | does considering a smaller subset of features for each split help? |

In [7]:
from run_test import run_test

decision_tree_results = run_test(best_estimator, splits)
display(decision_tree_results['confusion_matrix'])
print(decision_tree_results['classification_report'])

,predicted not depressed,predicted depressed
actual not depressed,351,0
actual depressed,0,9


Classification Report
               precision    recall  f1-score   support

not depressed       1.00      1.00      1.00       351
    depressed       1.00      1.00      1.00         9

     accuracy                           1.00       360
    macro avg       1.00      1.00      1.00       360
 weighted avg       1.00      1.00      1.00       360



### Interpretation of results
The `Decision Tree` performed exceptionally with `f1_score=0.97` for validation and `f1_score=1.0` for testing. In fact it performed much better than i originally expected.

With `max_depth=5` the model was able to almost perfectly discover the pattern from a single tree! It makes sense that a more shallow tree was not the most performant as the patterns seem to not be dominated heavily by individual features.

# Random Forest

In [8]:
from sklearn.ensemble import RandomForestClassifier

randForest = make_pipeline(preprocessor(remainder=remainder), RandomForestClassifier(criterion='entropy', n_jobs=-1, random_state=environment.Random_State))

randForest_cv, best_estimator, best_params = grid_cross_validation(randForest, X_train, y_train, param_grid={
    'randomforestclassifier__max_depth': [1, 3, 5, 10, 15, 20],
    'randomforestclassifier__n_estimators': [10, 50, 100],
})

display(best_params)
print("Best f1 score: ", randForest_cv.best_score_)

,Best Parameter
max_depth,10
n_estimators,50


Best f1 score:  0.3066666666666667


### Hyperparameter tuning
| Hyperparameter | Reason of choice |
| -------------- | ---------------- |
| `max_depth` | which depth of tree is most suitable? i.e. is the pattern dominated by a lower or higher number of decisions |
| `n_estimators` | what number of samples yields the best results? |

In [9]:
random_forest_results = run_test(best_estimator, splits)
display(random_forest_results['confusion_matrix'])
print(random_forest_results['classification_report'])

,predicted not depressed,predicted depressed
actual not depressed,351,0
actual depressed,8,1


Classification Report
               precision    recall  f1-score   support

not depressed       0.98      1.00      0.99       351
    depressed       1.00      0.11      0.20         9

     accuracy                           0.98       360
    macro avg       0.99      0.56      0.59       360
 weighted avg       0.98      0.98      0.97       360



### Interpretation of results
As predicted, `Random Forest` doesnt really work for the logic of the problem. It only scored `f1_score=0.3` for validation and `f1_score=0.2` for testing which is pretty bad.

As mentioned before, this is due to the nature of aggregate ensemble methods and how they must vote for the class. This makes sense for problems with a meaningful `accuracy` metric, however this problem does not (remember the min accuracy is ~`0.97`)

**Note from the future ([see Feature Importance](#feature-importance))**: I have now noticed that most features of the dataset did not contribute to the predictions at all. This probabaly lowered the performance of the Random Forest as a lot of the random feature subsets may have had no effect on the problem at all.

# Gradient Boost

In [10]:
from sklearn.ensemble import GradientBoostingClassifier

GBoost = make_pipeline(preprocessor(remainder=remainder), GradientBoostingClassifier(random_state=environment.Random_State))

gboost_cv, best_estimator, best_params = grid_cross_validation(GBoost, X_train, y_train, param_grid={
    'gradientboostingclassifier__max_depth': [3, 5, 7, 10],
    'gradientboostingclassifier__n_estimators': [10, 100, 500, 1000],
    'gradientboostingclassifier__learning_rate': [0.01, 0.1, 0.2, 0.5]
})

display(best_params)
print("Best f1 score: ", gboost_cv.best_score_)

,Best Parameter
learning_rate,0.01
max_depth,3.00
n_estimators,1000.00


Best f1 score:  0.9492063492063492


### Hyperparameter tuning
| Hyperparameter | Reason of choice |
| -------------- | ---------------- |
| `max_depth` | which depth of tree is most suitable? i.e. is the pattern dominated by a lower or higher number of decisions |
| `n_estimators` | what number of samples yields the best results? |
| `learning_rate` | does learning the pattern need more care or speed to yield the best results? |

In [11]:
gradient_boosting_results = run_test(best_estimator, splits)
display(gradient_boosting_results['confusion_matrix'])
print(gradient_boosting_results['classification_report'])

,predicted not depressed,predicted depressed
actual not depressed,351,0
actual depressed,0,9


Classification Report
               precision    recall  f1-score   support

not depressed       1.00      1.00      1.00       351
    depressed       1.00      1.00      1.00         9

     accuracy                           1.00       360
    macro avg       1.00      1.00      1.00       360
 weighted avg       1.00      1.00      1.00       360



### Interpretation of results

Unfortunately, the `Gradient Boosting` model ended up performing slightly worse than the single `Decision Tree`. My interpretation of the cause is that the gradient adjustments lead to slightly more overfitting of the training set. 

It still performed very well, and scored `f1_score=0.95` for validation and `f1_score=1.0` for testing.

An interesting thing to note is that the cross validation found that a `max_depth=3` performed the best, in contrast to the `max_depth=5` for the single `Decision Tree`, and as expected from what I have learned from the lecture content, the best results from cross validation were a high `num_estimators` and relitively low `learning_rate`.

# Multi Layer Perceptron (Neural Network)

In [16]:
from sklearn.neural_network import MLPClassifier

mlp = make_pipeline(preprocessor(remainder=remainder), MLPClassifier(max_iter=1000, learning_rate='adaptive', random_state=environment.Random_State))

mlp_cv, best_estimator, best_params = grid_cross_validation(mlp, X_train, y_train, param_grid={
    'mlpclassifier__hidden_layer_sizes': [(50, 50), (100, 50), (100, 100)],
    'mlpclassifier__learning_rate_init': [0.001, 0.01, 0.1],
    'mlpclassifier__alpha': [0.001, 0.01, 0.05, 0.1],
    'mlpclassifier__early_stopping': [True, False]
})

display(best_params)
print("Best f1 score: ", mlp_cv.best_score_)

,Best Parameter
alpha,0.1
early_stopping,False
hidden_layer_sizes,"(100, 100)"
learning_rate_init,0.001


Best f1 score:  0.8031746031746032


### Hyperparameter tuning
| Hyperparameter | Reason of choice |
| -------------- | ---------------- |
| `hidden_layer_size` | finding a good hidden layer scheme is essential for neural networks |
| `learning_rate_init` | what initial learning rate should be used for gradient descent? (adam optimiser is enabled) |
| `alpha` | how much penalty should I give to large weights |

In [17]:
mlp_results = run_test(best_estimator, splits)
display(mlp_results['confusion_matrix'])
print(mlp_results['classification_report'])

,predicted not depressed,predicted depressed
actual not depressed,349,2
actual depressed,5,4


Classification Report
               precision    recall  f1-score   support

not depressed       0.99      0.99      0.99       351
    depressed       0.67      0.44      0.53         9

     accuracy                           0.98       360
    macro avg       0.83      0.72      0.76       360
 weighted avg       0.98      0.98      0.98       360



### Interpretation of results
Unfortunately, the neural network does not seem to be very beneficial for the problem. The validation score is pretty good at `f1_score=0.8`, however the test score is only `f1_score=0.53`.

My interpretation of this is that the imbalanced nature of the dataset has made the network overfit to `not depressed`. 

# To conclude this section...
I will compare the results of all the models and give my final remarks, as well as an analysis of the improtance of each feature to the problem

In [14]:
from baseline_model import baseline_model

baseline = make_pipeline(preprocessor(remainder=remainder), baseline_model)
baseline_cv, best_estimator, best_params = grid_cross_validation(baseline, X_train, y_train, param_grid={})
baseline_score = run_test(baseline.fit(X_train, y_train), splits)["f1_score"]

df = pd.DataFrame([
  ("Linear Regression", baseline_cv.best_score_, baseline_score),
  ("KNN", knn_cv.best_score_, knn_results['f1_score']),
  ("Decision Tree", decisionTree_cv.best_score_, decision_tree_results['f1_score']),
  ("Random Forest", randForest_cv.best_score_, random_forest_results['f1_score']),
  ("Gradient Boosting", gboost_cv.best_score_, gradient_boosting_results['f1_score']),
  ("Neural Network", mlp_cv.best_score_, mlp_results['f1_score']),
], columns=['Model', 'Best F1 Score (CV)', 'F1 Score (Test)']).sort_values(by=['F1 Score (Test)', 'Best F1 Score (CV)'], ascending=False).reset_index(drop=True)

display(df)
print("The winner is... 🥳🍾", df.iloc[0]['Model'], "🥳🍾")

,Model,Best F1 Score (CV),F1 Score (Test)
0,Decision Tree,0.977778,1.000000
1,Gradient Boosting,0.949206,1.000000
2,Linear Regression,0.710000,0.571429
3,Neural Network,0.803175,0.533333
4,Random Forest,0.306667,0.200000
5,KNN,0.080000,0.000000


The winner is... 🥳🍾 Decision Tree 🥳🍾


Surprisingly, the baseline model `Logistic Regression` beats out most models, even the `Neural Network`! The `Tree` / `Boosting` models definitely dominate this problem with `Decision Tree` performing the best overall. 

This is certainly not what I expected in the beginning, but it demonstrates that the more complex model is not always the right choice. It also demonstrates that the Machine Learning experimentation process must be followed the whole way through instead of just assumed.

# Feature Importance
Using the best model (`Decision Tree`), I will outline the importance of each individual feature to the result

In [15]:
best_estimator = decisionTree_cv.best_estimator_.fit(X_train, y_train)
def remove_prefix(feature_name):
    return feature_name.split("__")[-1]

feature_names = [remove_prefix(name) for name in best_estimator["columntransformer"].get_feature_names_out()]

display(pd.DataFrame(best_estimator["decisiontreeclassifier"].feature_importances_, index=feature_names, columns=["Importance"]).sort_values(by="Importance", ascending=False))

,Importance
daily_social_media_hours,0.312656
anxiety_level,0.258421
stress_level,0.216043
sleep_hours,0.212880
gender_female,0.000000
gender_male,0.000000
platform_usage_Both,0.000000
age,0.000000
social_interaction_level,0.000000
platform_usage_TikTok,0.000000


As it turns out, only `daily_social_media_hours`, `anxiety_level`, `stress_level`, and `sleep_hours` contribute to the predictions of the `Decision Tree`.

With this in mind, i decided to see what would happen if I reduced the dimensionality manually by pruning the irrelevant features and re-running the models

### Please follow these steps:
1. [Go back to the top](#come-back-to-this-at-the-end-of-this-notebook) and uncomment the line `# remainder="drop"`. This will allow the models to be run with only the relevant features

2. Run all cells again. (you may need to restart the kernal)

3. [Read the new results](#to-conclude-this-section)

4. Read the insights bellow

# Insights from Dimensionality Reduction
Removing redundant features has greatly improved the performance of some of the models, and decreased the performance of others, and has changed the ranking.

| model | validation change | test change | updated rank |
| -- | -- | -- | -- |
| KNN | 0.08 -> 0.63 | 0.0 -> 0.36 | 6th |
| Linear Regression | 0.75 -> 0.63 | 0.57 -> 0.46 | 3rd -> 5th |
| Neural Network | 0.80 -> 0.71 | 0.58 -> 0.625 | 4th |
| Random Forest | 0.3 -> 0.8 | 0.2 -> 0.88 | 5th -> 3rd |
| Gradient Boosting | no change | no change | 2nd |
| Decision Tree | no change | no change | 1st |

The most substantial improvements were `KNN` and `Random Forest`. 
- `Random Forest` moved into the number 3 spot with a test score of `0.88`, a massive improvement! My interpretation of what happened was: with the redundant features gone, the samples could focus more on finding the pattern as the randomised feature subset per split is now guaranteed to have useful features.

- `KNN` is still the worst but its improvement was the best, up from `0.08` to `0.63` for validation! This is likely bc of the extra redundant dimensions destroying the distance calculations making it a very unreliable estimator.

Although the `Neural Network` had a decrease in performance, the validation and test scores have now evened out and it is less overfit.

# Next steps

My next goal is to polish the notebooks, and perhaps explore `clustering` and `pricipal component analysis`.